# Investment AI - Databricks Multi-Agent POC
## Notebook 00 - Environment & Workspace Validation

Before we create data, search indexes, semantic assets, tools or agents, this notebook verifies the Databricks workspace foundation.


### Why this matters

A production Agentic AI application depends on the platform around the model. We first validate Spark, Unity Catalog, Databricks SDK authentication and MLflow so later failures are easier to diagnose.


In [ ]:
import sys
import platform

print("Python version :", sys.version.split()[0])
print("Platform       :", platform.platform())
print("Notebook ready : ✅")

## 1. Validate Spark

In [ ]:
spark_version = spark.version
test_count = spark.range(10).count()
print("Spark version :", spark_version)
print("Spark test    :", test_count)
print("Spark         : ✅")

## 2. Validate Unity Catalog

In [ ]:
catalogs = spark.sql("SHOW CATALOGS")
display(catalogs)
catalog_names = [row["catalog"] for row in catalogs.collect()]
print("Catalogs visible:", catalog_names)
print("workspace catalog:", "✅" if "workspace" in catalog_names else "⚠️ not visible")

## 3. Validate Databricks SDK authentication

In [ ]:
from databricks.sdk import WorkspaceClient
workspace_client = WorkspaceClient()
current_user = workspace_client.current_user.me()
print("Authenticated user :", current_user.user_name)
print("Workspace host     :", workspace_client.config.host)
print("Databricks SDK     : ✅")

## 4. Validate MLflow

In [ ]:
import mlflow
mlflow.set_tracking_uri("databricks")
print("MLflow version      :", mlflow.__version__)
print("MLflow tracking URI :", mlflow.get_tracking_uri())
print("MLflow              : ✅")

## 5. Inspect agent-related libraries

Optional libraries may not be installed yet. We introduce them only when the corresponding capability is built.

In [ ]:
from importlib import import_module, metadata
packages = {
    "databricks-sdk": "databricks.sdk",
    "mlflow": "mlflow",
    "openai": "openai",
    "openai-agents": "agents",
    "databricks-openai": "databricks_openai",
    "databricks-mcp": "databricks_mcp",
    "mcp": "mcp",
}
results=[]
for package_name, module_name in packages.items():
    try:
        import_module(module_name)
        try: version=metadata.version(package_name)
        except Exception: version="installed"
        status="✅"
    except Exception:
        version="not installed"
        status="⚠️"
    results.append({"package":package_name,"version":version,"status":status})
display(spark.createDataFrame(results))

## 6. Environment summary

In [ ]:
required_checks = {
    "Spark": test_count == 10,
    "Unity Catalog workspace": "workspace" in catalog_names,
    "Databricks SDK": bool(current_user.user_name),
    "MLflow": mlflow.get_tracking_uri() == "databricks",
}
for name, passed in required_checks.items():
    print(f"{'✅' if passed else '❌'} {name}")
assert all(required_checks.values())
print("\n✅ Investment AI Databricks environment is ready.")

## Next

Notebook 01 will create the governed Unity Catalog foundation defined in `config/project.yaml`.